### RideWide --  Data Preprocessing 

Fixing All Data Quality issues identified during our EDA prepare clean dataset for feature engineering and Modelling

In [12]:
### setup   import Libraries

import pandas as pd 
import numpy as np
import seaborn as sns 
import matplotlib.pyplot as plt
import os, warnings



warnings.filterwarnings("ignore")
%matplotlib inline
sns.set_theme(style="whitegrid", font_scale=1.2)


### Define the location of our raw and processed
DATA_RAW = os.path.join("..","data","raw")
DATA_PROCESSED = os.path.join("..","data","processed")
os.makedirs(DATA_PROCESSED, exist_ok=True)


#### RIDEWISE_LONDON-MUHAMMAD
-data/
  - raw/
  - processed/

In [17]:
# Loading the datasets

riders = pd.read_csv(os.path.join(DATA_RAW,"riders.csv"), parse_dates=["signup_date"])
trips = pd.read_csv(os.path.join(DATA_RAW,"trips.csv"))
drivers = pd.read_csv(os.path.join(DATA_RAW,"drivers (1).csv"))
sessions = pd.read_csv(os.path.join(DATA_RAW,"sessions.csv"))
promotions = pd.read_csv(os.path.join(DATA_RAW,"promotions (1).csv"))

######

### \Step 1 Cleaning Exercise (Riders Dataset)

In [18]:
riders.head()

,user_id,signup_date,loyalty_status,age,city,avg_rating_given,churn_prob,referred_by
0,R00000,2025-01-24,Bronze,34.729629,Nairobi,5.0,0.142431,R00001
1,R00001,2024-09-09,Bronze,34.571020,Nairobi,4.7,0.674161,NaN
2,R00002,2024-09-07,Bronze,47.133960,Lagos,4.2,0.510379,NaN
3,R00003,2025-03-17,Bronze,41.658628,Nairobi,4.9,0.244779,NaN
4,R00004,2024-08-20,Silver,40.681709,Lagos,3.9,0.269960,R00002


In [ ]:
# Make  a copy of the original dataframes to work on 

riders_clean = riders.copy()


# Step 1a  ---- Fix age (round up the values )
riders_clean["age"] = riders_clean["age"].round().astype("int")

# Step 1b ----- Fix Churn label (converting from Continuous prob to binary label
# e.g (0.124, 0.4358 etc)
# e.g (1 & 0)

# step 1b(i)---------- create a threshold (0.5 - 50%)
# if churn probaility is >50 = churned (1)
# if churn probaility is <50 = not churned (0)

riders_clean ["churned"] = (riders_clean["churn_prob"] > 0.5).astype(int)
riders_clean = riders_clean.drop(columns=["churn_prob"])



# step 1c --- fix the referred by 
riders_clean["was_referred"] = riders_clean["referred_by"].notna().astype(int)
riders_clean = riders_clean.drop(columns=["referred_by"])

# option fill with "unknown" or "Not referred"
# option 2 return the ones that were referred (binary signal instead of racking my head on who was referred or not )

riders_clean.head()

,user_id,signup_date,loyalty_status,age,city,avg_rating_given,referred_by,churned,was_referred
0,R00000,2025-01-24,Bronze,35,Nairobi,5.0,R00001,0,1
1,R00001,2024-09-09,Bronze,35,Nairobi,4.7,NaN,1,0
2,R00002,2024-09-07,Bronze,47,Lagos,4.2,NaN,1,0
3,R00003,2025-03-17,Bronze,42,Nairobi,4.9,NaN,0,0
4,R00004,2024-08-20,Silver,41,Lagos,3.9,R00002,0,1


In [23]:
## check for formatting
print(f"Riders columns data types:\n{riders_clean.dtypes}\n")

Riders columns data types:
user_id                     object
signup_date         datetime64[ns]
loyalty_status              object
age                          int64
city                        object
avg_rating_given           float64
referred_by                 object
churned                      int64
was_referred                 int64
dtype: object



In [24]:
# drop referred by column since we have created a binary signal for it
riders_clean = riders_clean.drop(columns=["referred_by"], axis=1)

In [25]:
riders_clean.head()

,user_id,signup_date,loyalty_status,age,city,avg_rating_given,churned,was_referred
0,R00000,2025-01-24,Bronze,35,Nairobi,5.0,0,1
1,R00001,2024-09-09,Bronze,35,Nairobi,4.7,1,0
2,R00002,2024-09-07,Bronze,47,Lagos,4.2,1,0
3,R00003,2025-03-17,Bronze,42,Nairobi,4.9,0,0
4,R00004,2024-08-20,Silver,41,Lagos,3.9,0,1


### Step 2 - Cleaning Exercise (Trips)

In [26]:
#  copy the trips dataset
trips_clean = trips.copy()

## Step 2a ---- fix the date columns (convert to datetime format)(drop_off and pickup time)

for col in ["pickup_time", "dropoff_time"]:
    trips_clean[col] = (pd.to_datetime(trips_clean[col], utc=True, errors="coerce").dt.tz_localize(None))

trips_clean = trips_clean.dropna(subset=["pickup_time", "dropoff_time"])

# print out all attributes
print (f"Trips columns data types:\n{trips_clean.dtypes}\n")
trips_clean.head()

Trips columns data types:
trip_id                     object
user_id                     object
driver_id                   object
fare                       float64
surge_multiplier           float64
tip                        float64
payment_type                object
pickup_time         datetime64[ns]
dropoff_time        datetime64[ns]
pickup_lat                 float64
pickup_lng                 float64
dropoff_lat                float64
dropoff_lng                float64
weather                     object
city                        object
loyalty_status              object
dtype: object



,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,weather,city,loyalty_status
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 16:14:50,2024-11-27 17:06:50,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze
1,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 22:59:48,2024-10-28 23:12:48,6.675266,3.515740,6.641734,3.525620,Sunny,Lagos,Gold
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 03:09:41,2025-02-17 03:25:41,-1.248589,37.010668,-1.273182,37.018586,Cloudy,Nairobi,Bronze
3,T000003,R09573,D02657,16.43,1.0,0.01,Mobile Money,2024-06-18 17:22:14,2024-06-18 17:27:14,29.819554,31.188780,29.837689,31.232978,Cloudy,Cairo,Bronze
4,T000004,R03446,D01026,8.70,1.0,1.06,Card,2024-10-05 07:31:16,2024-10-05 08:01:16,-1.676479,36.729219,-1.638395,36.694063,Sunny,Nairobi,Gold


In [27]:
# Engineered some new features from the trips dataset

# 1. Trips Duration (in minutes )
trips_clean["trip_duration"] = (trips_clean["dropoff_time"] - trips_clean["pickup_time"]).dt.total_seconds() / 60

# 2. Total rervenue (fare = surge_multiplier + tip)
trips_clean["total_revenue"]=(trips_clean["fare"] * trips_clean["surge_multiplier"] + trips_clean["tip"])

#3. Hour of the day 
trips_clean["hour_of_day"] = trips_clean["pickup_time"].dt.hour

# 4. Day of the week
trips_clean["day_of_week"] = trips_clean["pickup_time"].dt.day_name()

In [28]:
# Data Quality in check
trips_clean = trips_clean[trips_clean["trip_duration"] > 0] # remove trips with non-positive duration
trips_clean["tip"] = trips_clean["tip"].fillna(0) # fill missing tips with 0
trips_clean["weather"] = trips_clean["weather"].fillna("unknown")

print(f"Rows shape: {trips_clean.shape}")
trips_clean.head

Rows shape: (200000, 20)


<bound method NDFrame.head of         trip_id user_id driver_id   fare  surge_multiplier   tip  \
0       T000000  R05207    D00315  12.11               1.0  0.00   
1       T000001  R09453    D03717   8.73               1.0  0.02   
2       T000002  R00567    D02035  19.68               1.0  0.00   
3       T000003  R09573    D02657  16.43               1.0  0.01   
4       T000004  R03446    D01026   8.70               1.0  1.06   
...         ...     ...       ...    ...               ...   ...   
199995  T199995  R08022    D04562  26.79               1.3  0.00   
199996  T199996  R05421    D03984  14.65               1.0  0.00   
199997  T199997  R06619    D01173  12.87               1.2  0.00   
199998  T199998  R02867    D00974  17.18               1.3  0.00   
199999  T199999  R07749    D04894  13.47               1.0  0.00   

        payment_type         pickup_time        dropoff_time  pickup_lat  \
0               Card 2024-11-27 16:14:50 2024-11-27 17:06:50   -1.108123   
1

### Step 3 Cleaning Exercise for Drivers dataset


In [29]:
drivers_clean = drivers.copy()
drivers_clean.head()

,driver_id,rating,vehicle_type,signup_date,last_active,city,acceptance_rate
0,D00000,3.1,SUV,2025-01-20,2025-01-06 18:23:09.312275,Cairo,0.679555
1,D00001,5.0,Sedan,2023-03-27,2025-04-27 01:44:02.472554,Nairobi,0.548786
2,D00002,4.5,Motorcycle,2024-05-02,2025-03-07 19:24:46.367672,Nairobi,0.593724
3,D00003,5.0,Motorcycle,2023-04-16,2025-03-26 19:16:24.253793,Nairobi,0.990000
4,D00004,4.4,Motorcycle,2023-05-28,2025-04-08 18:54:44.649615,Lagos,0.519773


In [30]:
print(drivers_clean.dtypes)
print(drivers_clean.isnull().sum())

driver_id           object
rating             float64
vehicle_type        object
signup_date         object
last_active         object
city                object
acceptance_rate    float64
dtype: object
driver_id          0
rating             0
vehicle_type       0
signup_date        0
last_active        0
city               0
acceptance_rate    0
dtype: int64


In [31]:
drivers_clean["rating"]= drivers_clean["rating"].fillna(drivers_clean["rating"].median())
drivers_clean["acceptance_rate"] = drivers_clean["acceptance_rate"].fillna(drivers_clean["acceptance_rate"].median())

### Step 4 CLeaning EXercise ; Session Dataset

In [ ]:
sessions_clean = sessions.copy()

# step 4(1)--- fix the session time date format
sessions_clean["session_time"] = pd.to_datetime(sessions_clean["session_time"], utc=True, errors="coerce").dt.tz_localize(None)



,session_id,rider_id,session_time,time_on_app,pages_visited,converted,city,loyalty_status
0,S000000,R08605,2025-04-27 18:57:06+02:05,79,4,1,Cairo,Bronze
1,S000001,R08823,2025-04-27 07:32:22+02:27,101,3,0,Nairobi,Silver
2,S000002,R05342,2025-04-27 23:17:25+02:05,12,1,0,Cairo,Bronze
3,S000003,R05057,2025-04-27 14:40:25+00:14,19,1,0,Lagos,Silver
4,S000004,R09614,2025-04-27 08:31:22+00:14,4,1,0,Lagos,Bronze


In [36]:
# step 4(2)----- drop rows with invalid session_time

sessions_clean = sessions_clean.dropna(subset=["session_time"])

print(f"Sessions_clean : {sessions_clean.shape}")

sessions_clean.head()

Sessions_clean : (50000, 8)


,session_id,rider_id,session_time,time_on_app,pages_visited,converted,city,loyalty_status
0,S000000,R08605,2025-04-27 16:52:06,79,4,1,Cairo,Bronze
1,S000001,R08823,2025-04-27 05:05:22,101,3,0,Nairobi,Silver
2,S000002,R05342,2025-04-27 21:12:25,12,1,0,Cairo,Bronze
3,S000003,R05057,2025-04-27 14:26:25,19,1,0,Lagos,Silver
4,S000004,R09614,2025-04-27 08:17:22,4,1,0,Lagos,Bronze


### STEP 5--- REFERENTIAL INTEGRITY

In [38]:
valid_riders = set(riders_clean["user_id"])
valid_drivers = set(drivers_clean["driver_id"])


# check the trips and sessions datasset
before_trips = len(trips_clean)
before_sessions = len(sessions_clean)

# filter to check if riders are in trips dataset
trips_clean = trips_clean[trips_clean["user_id"].isin(valid_riders)]
trips_clean = trips_clean[trips_clean["driver_id"].isin(valid_drivers)]
sessions_clean = sessions_clean[sessions_clean["rider_id"].isin(valid_riders)]


# check the descripancies
print(f"Trips that were dropped: {before_trips - len(trips_clean)}")
print(f"Sessions that were dropped: {before_sessions - len(sessions_clean)}")
print("Everything looks good")

Trips that were dropped: 0
Sessions that were dropped: 0
Everything looks good


### Validate & Save 

In [49]:
print ("Row count = raw vs clean")
for name, raw, clean in [
    ("Riders", riders, riders_clean),
    ("Trips", trips, trips_clean),
    ("Drivers", drivers, drivers_clean),
    ("Sessions", sessions, sessions_clean),
    ("Promotions", promotions, promotions)
]:
    print(f"{name:<10} : {len(raw):>7,} vs {len(clean):>7,}")

Row count = raw vs clean
Riders     :  10,000 vs  10,000
Trips      : 200,000 vs 200,000
Drivers    :   5,000 vs   5,000
Sessions   :  50,000 vs  50,000
Promotions :      20 vs      20


In [52]:
files = {
    "riders_clean.csv": riders_clean,
    "trips_clean.csv": trips_clean,
    "drivers_clean.csv": drivers_clean,
    "sessions_clean.csv": sessions_clean
}

for fname , df in files.items():
    df.to_csv(os.path.join(DATA_PROCESSED, fname), index=False)
    print("Data saved successfully")

Data saved successfully
Data saved successfully
Data saved successfully
Data saved successfully
